> ⚠️ Note: Sensitive credentials have been replaced with placeholders for security purposes.

# Kronos Timecard Data Pipeline (Microsoft Fabric)

## Overview
This notebook implements an automated data pipeline that extracts employee timecard (punch) data from the Kronos API, processes it into structured shift-level records, and stores the results in a Microsoft Fabric Lakehouse table.

The pipeline supports both:
- Historical backfill (last 30 days)
- Incremental daily updates

---

## Architecture
Kronos API → Fabric Notebook → Lakehouse (Delta Table) → Power BI

---

## Key Features
- OAuth-based API authentication
- Incremental data ingestion
- Deduplication using Delta MERGE (upsert)
- Shift reconstruction (clock in/out inference)
- Break time handling
- Audit column for data tracking (`loaded_at`)

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

'''
NOTE:
Credentials are stored securely in production (e.g. Azure Key Vault).
Placeholders are used here for demonstration purposes.
'''

# Kronos API config
hostname = "<Kronos_API_Hostname>"
client_id = "<Client_ID>"
client_secret = "<Client_Secret>"
username = "<API_Username>"
password = "<API_Password>"

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 3, Finished, Available, Finished, False)

## Data Source: Kronos API

The data is sourced from the Kronos Workforce Management API.

The API returns employee punch timestamps but does not explicitly label:
- Clock in
- Clock out

---

## Data Transformation Logic

To reconstruct shifts:

- The **first punch** of a sequence is treated as **Clock IN**
- The **last punch** is treated as **Clock OUT**
- Punches are sorted chronologically
- A gap of >12 hours indicates a new shift

---

## Break Handling

Breaks are identified using labor categories:
- `GXO.Break` → Break start
- `GXO.Working` → Break end

In [2]:
def get_access_token():
    token_url = f"https://{hostname}/api/authentication/access_token"

    data = {
        'username': username,
        'password': password,
        'client_id': client_id,
        'client_secret': client_secret,
        'grant_type': 'password',
        'auth_chain': 'OAuthLdapService'
    }

    response = requests.post(token_url, data=data)
    response.raise_for_status()

    return response.json().get("access_token")


def call_api(report_date):
    token = get_access_token()

    start_date = report_date
    end_date = (datetime.strptime(report_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")

    url = f"https://{hostname}/api/v1/timekeeping/timecard/multi_read"

    payload = {
        "where": {
            "dateRange": {
                "startDate": start_date,
                "endDate": end_date
            },
            "hyperFind": { "qualifier": "21.API_PunchData" }
        },
        "select": ["PUNCHES"]
    }

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()

    return response.json()

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 4, Finished, Available, Finished, False)

## Deduplication Strategy

To prevent duplicate records, the pipeline uses a Delta Lake **MERGE (upsert)** operation.

### Unique Key:
- `employee_id + clock_in`

### Behaviour:
- New records → INSERT
- Existing records → UPDATE
- Duplicate loads → ignored safely

This ensures:
- Idempotent processing (safe to rerun)
- No duplicate shifts
- Data consistency over time

In [3]:
def process_timecard_data(response_data):
    all_shifts = []

    for employee_data in response_data:
        punches = employee_data.get('punches', [])
        if not punches:
            continue

        if 'employee' not in punches[0]:
            continue

        employee_id = punches[0]['employee'].get('qualifier')

        processed = []

        for punch in punches:
            dt = pd.to_datetime(punch['punchDtm'])

            punch_type = "CLOCK"

            if 'laborCategories' in punch:
                if punch['laborCategories']['laborString'] == 'GXO.Break':
                    punch_type = 'BREAK_IN'
                elif punch['laborCategories']['laborString'] == 'GXO.Working':
                    punch_type = 'BREAK_OUT'

            processed.append({
                "employee_id": employee_id,
                "datetime": dt,
                "type": punch_type
            })

        processed = sorted(processed, key=lambda x: x["datetime"])

        current_shift = []

        for i, p in enumerate(processed):
            current_shift.append(p)

            is_last = i == len(processed) - 1
            is_new_shift = not is_last and (processed[i+1]["datetime"] - p["datetime"]) > timedelta(hours=12)

            if is_last or is_new_shift:
                if len(current_shift) >= 2:
                    shift = {
                        "employee_id": employee_id,
                        "clock_in": current_shift[0]["datetime"],
                        "clock_out": current_shift[-1]["datetime"],
                        "break_in": None,
                        "break_out": None
                    }

                    for sp in current_shift:
                        if sp["type"] == "BREAK_IN":
                            shift["break_in"] = sp["datetime"]
                        elif sp["type"] == "BREAK_OUT":
                            shift["break_out"] = sp["datetime"]

                    all_shifts.append(shift)

                current_shift = []

    df = pd.DataFrame(all_shifts)

    return df

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 5, Finished, Available, Finished, False)

## Processing Steps

1. Authenticate with Kronos API
2. Retrieve punch data for a given date
3. Process raw punches into structured shifts
4. Calculate break times and working hours
5. Convert data into Spark DataFrame
6. Upsert into Lakehouse Delta table

---

## Output Table: `kronos_timecard`

Columns include:
- employee_id
- clock_in
- clock_out
- break_in
- break_out
- loaded_at (audit column)

In [ ]:
def upsert_to_lakehouse(df):
    if df.empty:
        print("No data to save")
        return

    from pyspark.sql.functions import col, current_timestamp

    spark_df = spark.createDataFrame(df)

    spark_df = spark_df \
        .withColumn("employee_id", col("employee_id").cast("string")) \
        .withColumn("clock_in", col("clock_in").cast("timestamp")) \
        .withColumn("clock_out", col("clock_out").cast("timestamp")) \
        .withColumn("break_in", col("break_in").cast("timestamp")) \
        .withColumn("break_out", col("break_out").cast("timestamp"))

    # AUDIT COLUMN
    spark_df = spark_df.withColumn("loaded_at", current_timestamp())

    # DEDUPLICATION
    spark_df = spark_df.dropDuplicates(["employee_id", "clock_in"])

    # temp view
    spark_df.createOrReplaceTempView("temp_view")

    table_name = "kronos_timecard"

    if spark.catalog.tableExists(table_name):

        spark.sql(f"""
        MERGE INTO {table_name} target
        USING temp_view source
        ON target.employee_id = source.employee_id
        AND target.clock_in = source.clock_in
        
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """)

        print("✅ Upsert completed (no duplicates)")

    else:
        spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print("✅ Table created")

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 6, Finished, Available, Finished, False)

## Execution Modes

### 🔹 Backfill Mode
- Pulls last 30 days of data
- Used for initial load only

### 🔹 Daily Mode
- Pulls previous day only
- Runs via scheduled pipeline
- Uses MERGE to avoid duplicates

---

## Scheduling

The notebook is triggered daily using a Microsoft Fabric Pipeline.

In [5]:
'''
def main_backfill():
    print("Running FULL 30-day backfill...")

    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)

    all_data = []

    current_date = start_date

    while current_date <= end_date:
        report_date = current_date.strftime('%Y-%m-%d')
        print(f"Processing {report_date}")

        try:
            response = call_api(report_date)
            df = process_timecard_data(response)

            if not df.empty:
                all_data.append(df)

        except Exception as e:
            print(f"Error on {report_date}: {e}")

        current_date += timedelta(days=1)

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        upsert_to_lakehouse(final_df)

    print("✅ Backfill complete")
'''

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 7, Finished, Available, Finished, False)

In [6]:
# main_backfill()

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 8, Finished, Available, Finished, False)

Running FULL 30-day backfill...
Processing 2026-05-02
Processing 2026-05-03
Processing 2026-05-04
Processing 2026-05-05
Processing 2026-05-06
Processing 2026-05-07
Processing 2026-05-08
Processing 2026-05-09
Processing 2026-05-10
Processing 2026-05-11
Processing 2026-05-12
Processing 2026-05-13
Processing 2026-05-14
Processing 2026-05-15
Processing 2026-05-16
Processing 2026-05-17
Processing 2026-05-18
Processing 2026-05-19
Processing 2026-05-20
Processing 2026-05-21
Processing 2026-05-22
Processing 2026-05-23
Processing 2026-05-24
Processing 2026-05-25
Processing 2026-05-26
Processing 2026-05-27
Processing 2026-05-28
Processing 2026-05-29
Processing 2026-05-30
Processing 2026-05-31
Processing 2026-06-01
✅ Table created
✅ Backfill complete


In [7]:
# spark.sql("SELECT COUNT(*) FROM kronos_timecard").show()

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 9, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|   41695|
+--------+



In [8]:
# spark.sql("SELECT * FROM kronos_timecard LIMIT 10").show()

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 10, Finished, Available, Finished, False)

+-----------+-------------------+-------------------+-------------------+-------------------+--------------------+
|employee_id|           clock_in|          clock_out|           break_in|          break_out|           loaded_at|
+-----------+-------------------+-------------------+-------------------+-------------------+--------------------+
|   11610231|2026-05-02 05:48:00|2026-05-02 14:00:00|2026-05-02 11:39:00|2026-05-02 12:10:00|2026-06-01 11:05:...|
|   11610223|2026-05-02 13:49:00|2026-05-02 22:00:00|2026-05-02 17:53:00|2026-05-02 18:24:00|2026-06-01 11:05:...|
|   11610238|2026-05-02 13:42:00|2026-05-03 16:00:00|2026-05-03 10:03:00|2026-05-03 10:33:00|2026-06-01 11:05:...|
|   11610239|2026-05-02 01:38:00|2026-05-02 06:00:00|2026-05-02 01:38:00|2026-05-02 02:10:00|2026-06-01 11:05:...|
|   11610239|2026-05-02 21:55:00|2026-05-03 06:00:00|2026-05-03 01:37:00|2026-05-03 02:09:00|2026-06-01 11:05:...|
|   30275859|2026-05-02 05:53:00|2026-05-02 14:01:00|2026-05-02 10:20:00|2026-05

In [9]:
def main_daily():
    print("Running DAILY load...")

    yesterday = datetime.now() - timedelta(days=1)
    report_date = yesterday.strftime('%Y-%m-%d')

    response = call_api(report_date)
    df = process_timecard_data(response)

    if df.empty:
        print("No data returned")
        return

    # ✅ Use UPSERT instead of append
    upsert_to_lakehouse(df)

    print("✅ Daily load complete")

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 11, Finished, Available, Finished, False)

In [10]:
main_daily()

StatementMeta(, c152d0bc-8dd3-4d96-942a-972cc98fc8ec, 12, Finished, Available, Finished, False)

Running DAILY load...
✅ Upsert completed (no duplicates)
✅ Daily load complete


## Future Improvements

- Partition table by date for performance optimization
- Move credentials to Azure Key Vault
- Implement retry logic for API failures
- Add incremental Power BI refresh
- Introduce data validation checks

---

## Technologies Used

- Python (requests, pandas)
- PySpark (Fabric)
- Microsoft Fabric Lakehouse
- Delta Tables
- Power BI